In [0]:
from pyspark.sql.functions import col, when, try_to_timestamp, coalesce, lit, create_map, decode, trim

# criação do schema no unity catalog,
spark.sql("CREATE SCHEMA IF NOT EXISTS foodDelivery.silver")

# 1. carrega os dados da bronze
df_bronze = spark.table('foodDelivery.staging.orders_synthetic')

# 2. calcula a media para inputação (Jarak_Kirim_KM)
median_distance = df.approxQuantile("Jarak_Kirim_KM", [0.5], 0.01)[0]

# 3. Dicionários de Tradução (Mapas)
# Status do Pedido (Status_Pesanan)
status_map = create_map([
    lit("Selesai"), lit("Concluído"),
    lit("Dibatalkan"), lit("Cancelado"),
    lit("Sedang Dikirim"), lit("Em Entrega"),
    lit("Refund"), lit("Reembolsado")
])

# Nível de Reclamação (Tingkat_Keluhan)
complaint_map = create_map([
    lit("Tidak Ada"), lit("Nenhuma"),
    lit("Rendah"), lit("Baixa"),
    lit("Sedang"), lit("Média"),
    lit("Tinggi"), lit("Alta")
])

# menu
menu_map = create_map([
    lit("Kopi"), lit("Café"),
    lit("Mie"), lit("Macarrão"),
    lit("Martabak"), lit("Martabak (Panqueca Indonésia)"),
    lit("Ayam"), lit("Frango")
])

# 4.Transformação final
df_silver = df_bronze.select(
        col("ID_Pesanan").alias("order_id"),

        # Padronização de data
        when(col("Waktu_Transaksi").contains("/"), 
             to_timestamp(trim(col("Waktu_Transaksi")), "dd/MM/yyyy HH:mm")
             ).otherwise(col("Waktu_Transaksi")
        ).alias("transaction_time"), 

        # tradução da categoria
        coalesce(menu_map[col("Kategori_Menu")], col("Kategori_Menu")).alias("menu_category"), 
        col("Harga_Pesanan").cast("double").alias("order_price"),

        # input de mediana (Jarak_Kirim_KM)
        coalesce(col("Jarak_Kirim_KM"), lit(median_distance)).alias("shipping_distance_km"),
        col("Waktu_Tunggu_Menit").alias("waiting_time_minutes"),
        col("Rating_Pelanggan").alias("customer_rating"),
        col("Ulasan_Teks").alias("review_text"),
        col("Status_Promo").alias("promo_status"),
        
        # Aplicação das Traduções
        coalesce(complaint_map[col("Tingkat_Keluhan")], col("Tingkat_Keluhan")).alias("complaint_level"),
        coalesce(status_map[col("Status_Pesanan")], col("Status_Pesanan")).alias("order_status")
) 

# 5.Salvar na camada Silver
(df_silver.write
 .format('delta')
 .mode('overwrite')
 .saveAsTable('foodDelivery.silver.orders'))